# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [25]:
import os
import boto3
from dotenv import load_dotenv
import sys
import scipy.sparse
from IPython.display import display
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


load_dotenv()

True

In [26]:
import utils
import importlib

importlib.reload(utils)

<module 'utils' from '/home/mle-user/sprint5/final_project/utils.py'>

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

In [27]:
import gc
import numpy as np
import pandas as pd

K = 10

In [28]:
import boto3, os
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket_name = os.getenv("S3_BUCKET_NAME")

In [29]:
candidates = pd.read_parquet(
    "candidates_result.parquet"
)

KeyboardInterrupt: 

In [ ]:
from catboost import CatBoostRanker

model_path = "services/models/catboost_ranker.cbm"

ranker = CatBoostRanker()

ranker.load_model(
    model_path
)

print("Модель успешно загружена")

Модель успешно загружена


In [31]:
s3.upload_file(
    model_path,
    bucket_name,
    "recsys/models/catboost_ranker.cbm"
)

print("Модель загружена в S3")

Модель загружена в S3


In [23]:
FEATURES = [
    "als_score",
    "als_rank",
    "similarity_score",
    "similarity_rank",
    "popular_score",
    "popular_rank",
]

test_predictions = ranker.predict(
    candidates[FEATURES]
)

print(test_predictions[:10])

[-0.8887607  -0.06446636 -0.04783882 -0.9243855  -0.43771442  0.13178829
 -0.89423998 -0.02725313 -0.40994465 -0.54144935]


In [7]:
import pyarrow.parquet as pq

schema = pq.read_schema(
    "recommendations/similar/similar.parquet"
)

print(schema.names)

['itemid', 'similar_itemid', 'score', 'rank']


In [8]:
# ============================================================
# ЗАГРУЗКА РЕКОМЕНДАЦИЙ ДЛЯ ОЦЕНКИ
# ============================================================

top_popular = pd.read_parquet(
    "recommendations/top_popular/top_popular.parquet",
    columns=["itemid", "score", "rank"]
)

top_popular = top_popular.rename(
    columns={
        "itemid": "item_id",
        "score": "popular_score",
        "rank": "popular_rank"
    }
)

top_popular = top_popular[
    top_popular["popular_rank"] <= K
].copy()


personal_als = pd.read_parquet(
    "recommendations/personal_als/personal_als.parquet",
    columns=[
        "visitorid",
        "itemid",
        "score",
        "rank",
    ],
    filters=[
        ("rank", "<=", K)
    ]
)

personal_als = personal_als.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
        "score": "als_score",
        "rank": "als_rank",
    }
)


recommendations = pd.read_parquet(
    "recommendations.parquet",
    columns=["user_id", "item_id", "rank"],
    filters=[("rank", "<=", K)]
)


print("Top Popular:", top_popular.shape)
print("ALS:", personal_als.shape)
print("Final:", recommendations.shape)

Top Popular: (10, 3)
ALS: (12803290, 4)
Final: (12803290, 3)


In [9]:
events = pd.read_parquet(
    "parquet_cleaned/events.parquet"
)


In [10]:
events.head(1)

,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,<NA>


In [11]:
events_rank = events.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
    }
)


In [12]:
# Приводим timestamp к datetime, если ещё не приводили
events["event_time"] = pd.to_datetime(
    events["timestamp"]
)

# Конец датасета
data_end = (
    events["event_time"].max()
    + pd.Timedelta(milliseconds=1)
)

# Последние 7 дней -> финальная оценка
eval_start = (
    data_end
    - pd.Timedelta(days=7)
)

# Предыдущие 7 дней -> обучение ranker
ranker_start = (
    eval_start
    - pd.Timedelta(days=7)
)

print("ranker_start:", ranker_start)
print("eval_start:", eval_start)
print("data_end:", data_end)

ranker_start: 2015-09-04 02:59:47.789000
eval_start: 2015-09-11 02:59:47.789000
data_end: 2015-09-18 02:59:47.789000


In [13]:
# ============================================================
# GROUND TRUTH
# ============================================================

events_eval = events[
    ["visitorid", "itemid", "event", "timestamp"]
].copy()

events_eval = events_eval.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
        "timestamp": "event_time",
    }
)

events_eval["event_time"] = pd.to_datetime(
    events_eval["event_time"]
)

# Берём только финальный evaluation-период
events_eval = events_eval[
    (events_eval["event_time"] >= eval_start)
    &
    (events_eval["event_time"] < data_end)
].copy()

# В качестве релевантных действий используем те же события,
# что и при формировании target для ranker
events_eval = events_eval[
    events_eval["event"].isin(
        [
            "addtocart",
            "transaction",
        ]
    )
].copy()

ground_truth = (
    events_eval[
        [
            "user_id",
            "item_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("Ground truth:", ground_truth.shape)
print(
    "Период:",
    events_eval["event_time"].min(),
    "->",
    events_eval["event_time"].max()
)


Ground truth: (2841, 2)
Период: 2015-09-11 03:00:43.360000 -> 2015-09-18 02:50:49.834000


In [14]:
del events_eval
gc.collect()

0

In [15]:
eval_users = np.intersect1d(
    ground_truth["user_id"].unique(),
    personal_als["user_id"].unique()
)

ground_truth = ground_truth[
    ground_truth["user_id"].isin(eval_users)
].copy()

print(
    "Пользователей для оценки:",
    len(eval_users)
)

Пользователей для оценки: 242


In [16]:
popular_items = (
    top_popular
    .sort_values("popular_rank")
    ["item_id"]
    .to_numpy()
)

top_popular_eval = pd.DataFrame({
    "user_id": np.repeat(
        eval_users,
        len(popular_items)
    ),
    "item_id": np.tile(
        popular_items,
        len(eval_users)
    ),
    "rank": np.tile(
        np.arange(1, len(popular_items) + 1),
        len(eval_users)
    )
})

In [17]:
personal_als_eval = (
    personal_als[
        personal_als["user_id"].isin(eval_users)
    ][
        ["user_id", "item_id", "als_rank"]
    ]
    .rename(
        columns={"als_rank": "rank"}
    )
)

In [18]:
final_eval = recommendations[
    recommendations["user_id"].isin(eval_users)
][
    ["user_id", "item_id", "rank"]
].copy()

In [19]:
catalog = pd.read_parquet(
    "parquet_cleaned/item_properties.parquet",
    columns=["itemid"]
)

catalog_size = catalog["itemid"].nunique()

del catalog
gc.collect()

print("Размер каталога:", catalog_size)

Размер каталога: 417053


In [20]:
events_train = events[
    events["event_time"] < eval_start
][
    ["itemid"]
].copy()

item_counts = (
    events_train["itemid"]
    .value_counts()
)

total_interactions = len(events_train)

novelty_map = -np.log2(
    item_counts / total_interactions
)

del events_train
del item_counts

gc.collect()

0

In [21]:
catalog = pd.read_parquet(
    "parquet_cleaned/item_properties.parquet",
    columns=["itemid"]
)

catalog_size = (
    catalog["itemid"]
    .nunique()
)

print(
    "Размер каталога:",
    catalog_size
)

del catalog
gc.collect()

Размер каталога: 417053


0

In [22]:
def evaluate_recommendations(
    recommendations_df,
    ground_truth,
    catalog_size,
    novelty_map,
    k=10
):
    recs = (
        recommendations_df[
            recommendations_df["rank"] <= k
        ][["user_id", "item_id"]]
        .drop_duplicates()
        .copy()
    )

    truth = (
        ground_truth[
            ["user_id", "item_id"]
        ]
        .drop_duplicates()
        .copy()
    )

    relevant_count = (
        truth
        .groupby("user_id")
        .size()
    )

    hits = recs.merge(
        truth.assign(hit=1),
        on=["user_id", "item_id"],
        how="left"
    )

    hits["hit"] = (
        hits["hit"]
        .fillna(0)
        .astype("int8")
    )

    hits_per_user = (
        hits
        .groupby("user_id")["hit"]
        .sum()
        .reindex(
            relevant_count.index,
            fill_value=0
        )
    )

    precision = (
        hits_per_user / k
    ).mean()

    recall = (
        hits_per_user
        /
        relevant_count
    ).mean()

    coverage = (
        recs["item_id"].nunique()
        /
        catalog_size
    )

    novelty = (
        recs["item_id"]
        .map(novelty_map)
        .dropna()
        .mean()
    )

    return {
        "precision": precision,
        "recall": recall,
        "coverage": coverage,
        "novelty": novelty
    }

In [23]:
metrics_top_popular = evaluate_recommendations(
    top_popular_eval,
    ground_truth,
    catalog_size,
    novelty_map,
    k=K
)

metrics_als = evaluate_recommendations(
    personal_als_eval,
    ground_truth,
    catalog_size,
    novelty_map,
    k=K
)

metrics_final = evaluate_recommendations(
    final_eval,
    ground_truth,
    catalog_size,
    novelty_map,
    k=K
)

In [24]:
metrics = pd.DataFrame([
    {
        "model": "Top Popular",
        **metrics_top_popular
    },
    {
        "model": "ALS",
        **metrics_als
    },
    {
        "model": "Final Ranker",
        **metrics_final
    }
]).set_index("model")

display(
    metrics.style.format({
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "coverage": "{:.6f}",
        "novelty": "{:.4f}"
    })
)

,precision,recall,coverage,novelty
model,,,,
Top Popular,0.002066,0.007505,0.000024,11.0312
ALS,0.009504,0.067424,0.001537,12.2737
Final Ranker,0.010744,0.077755,0.001942,12.5657


In [24]:
%reset -f

import gc
gc.collect()

0

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.

По результатам оценки персональные рекомендации ALS значительно превосходят неперсонализированный Top Popular по всем рассчитанным метрикам. Дополнительное ранжирование кандидатов с помощью CatBoostRanker позволило улучшить качество рекомендаций относительно базового ALS: precision@10 увеличился с 0.009504 до 0.010744, а recall@10 — с 0.067424 до 0.077755. Также увеличилось покрытие каталога с 0.001537 до 0.001942 и novelty с 12.2737 до 12.5657. Таким образом, итоговая ранжирующая модель позволяет точнее упорядочить объединённый набор кандидатов и одновременно повышает разнообразие рекомендуемых товаров относительно ALS.